# Q-Caliber: Quantum-Calibrated Graph Diffusion

**Q-Caliber** (Quantum-Calibrated Graph Diffusion) is a novel approach that uses *local* quantum-walk statistics from sampled subnetworks to calibrate parameters $\theta$ of a *global* graph diffusion operator $g_{\theta}(L)$. This enables scalable graph learning by combining quantum walk insights with classical diffusion methods.

## Key Innovation

Instead of running expensive quantum walks on the full graph, Q-Caliber:
1. Samples representative subnetworks (ego-nets)
2. Runs quantum walks (CTQW) on these small subgraphs using **Hiperwalk**
3. Fits global diffusion parameters (heat kernel or polynomial filter) to match quantum walk behavior
4. Applies the calibrated classical operator $g_{\theta}(L)$ to the full graph

This hybrid approach achieves quantum-informed diffusion at classical computational cost.

## Prerequisites

All required packages are in `requirements.txt`. Install with:
```bash
pip install -r requirements.txt
```

Key dependencies: `hiperwalk`, `numpy`, `scipy`, `networkx`, `matplotlib`, `scikit-learn`

## Notebook Outline

1. **Load graph**: Biological network (PPI/gene-gene) or synthetic graph
2. **Sample subnetworks**: Coverage-aware ego-net sampling using QuVINE
3. **Quantum walk simulation**: Run CTQW on subnetworks with QuVINE's Hiperwalk integration
4. **Parameter calibration**: Fit global filter parameters $\theta$ (heat kernel or polynomial)
5. **Apply & evaluate**: Use calibrated $g_{\theta}(L)$ for node classification/ranking

# Imports

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import networkx as nx
import matplotlib.pyplot as plt

# QuVINE imports
from quvine.data.subgraph import (
    expand_neighborhood,
    induce_subgraph_by_nodes,
    subsample_nodes_with_protected
)
from quvine.walks.ctqw import generate_ctqw_hiperwalk_scores
from quvine.utils.utilities import get_stats

# Try importing Hiperwalk
try:
    import hiperwalk as hpw
    HIPERWALK_AVAILABLE = True
except Exception as e:
    HIPERWALK_AVAILABLE = False
    print('Hiperwalk import failed. Install with: pip install hiperwalk')
    print('Error:', e)

# Optional sklearn
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, average_precision_score
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

## 1) Load (or generate) a graph

In practice, replace the toy graph below with your PPI adjacency (e.g., BioGRID/STRING) using QuVINE's `load_graph()` function.

For a quick prototype, we generate a graph and treat a small set of nodes as "seed" disease genes.

In [ ]:
# Toy graph generator (replace with real PPI using quvine.data.data_loader.load_graph)
np.random.seed(7)
N0 = 300
p = 0.02
G = nx.erdos_renyi_graph(N0, p, seed=7)
G = nx.convert_node_labels_to_integers(G)

# Ensure connectivity for convenience
if not nx.is_connected(G):
    largest = max(nx.connected_components(G), key=len)
    G = G.subgraph(largest).copy()
    # IMPORTANT: Relabel nodes to be contiguous integers starting from 0
    G = nx.convert_node_labels_to_integers(G, first_label=0)

N = G.number_of_nodes()
print('Nodes:', N, 'Edges:', G.number_of_edges())

# Create node-to-index mapping (nodes should already be 0 to N-1)
nodes_list = list(G.nodes())
node_to_idx = {n: i for i, n in enumerate(nodes_list)}
print(f'Node IDs range: {min(nodes_list)} to {max(nodes_list)}')

# Display graph statistics using QuVINE utility
stats = get_stats(G)
print('\nGraph Statistics:')
for key, value in stats.items():
    print(f'  {key}: {value}')

# Example seed genes (replace with disease seeds)
seed_nodes = np.random.choice(nodes_list, size=5, replace=False).tolist()
print('\nSeed nodes:', seed_nodes)

# Adjacency and Laplacian
A = nx.to_scipy_sparse_array(G, format='csr', dtype=float)
# combinatorial Laplacian L = D - A
deg = np.array(A.sum(axis=1)).reshape(-1)
L = sp.diags(deg) - A

## 2) Coverage-aware subnetwork sampling using QuVINE

We use QuVINE's `expand_neighborhood` and `induce_subgraph_by_nodes` functions to sample ego-nets around seeds and additional nodes via importance sampling.

In [ ]:
def get_ego_net_nodes_quvine(G, center, k=2, max_nodes=None):
    """
    Get ego-net nodes using QuVINE's expand_neighborhood function.
    
    Args:
        G: NetworkX graph
        center: Center node
        k: Hop radius
        max_nodes: Maximum number of nodes (optional truncation)
    
    Returns:
        List of nodes in the ego-net
    """
    # Use QuVINE's expand_neighborhood function
    nodes = expand_neighborhood(G, roots={center}, radius=k)
    nodes = list(nodes)
    
    if max_nodes is not None and len(nodes) > max_nodes:
        # Keep center + top-degree neighbors first
        nodes_sorted = sorted(nodes, key=lambda n: G.degree[n], reverse=True)
        if center in nodes_sorted:
            nodes_sorted.remove(center)
        nodes = [center] + nodes_sorted[:max_nodes-1]
    
    return nodes


def importance_sample_centers(G, m, exclude=set(), bins=6, seed=0):
    """
    Sample m centers without replacement, stratified by degree bins.
    
    Args:
        G: NetworkX graph
        m: Number of centers to sample
        exclude: Set of nodes to exclude
        bins: Number of degree bins for stratification
        seed: Random seed
    
    Returns:
        List of sampled center nodes
    """
    rng = np.random.default_rng(seed)
    nodes = np.array([n for n in G.nodes() if n not in exclude])
    degrees = np.array([G.degree[n] for n in nodes])

    qs = np.quantile(degrees, np.linspace(0, 1, bins+1))
    strata = []
    for i in range(bins):
        lo, hi = qs[i], qs[i+1]
        mask = (degrees >= lo) & (degrees <= hi) if i == bins-1 else (degrees >= lo) & (degrees < hi)
        strata.append(nodes[mask])

    per = int(np.ceil(m / bins))
    chosen = []
    for s in strata:
        if len(chosen) >= m:
            break
        if len(s) == 0:
            continue
        k = min(per, m - len(chosen), len(s))
        chosen.extend(rng.choice(s, size=k, replace=False).tolist())

    if len(chosen) < m:
        remaining = np.array([n for n in nodes if n not in set(chosen)])
        k = m - len(chosen)
        if len(remaining) > 0:
            chosen.extend(rng.choice(remaining, size=min(k, len(remaining)), replace=False).tolist())

    return chosen[:m]


# Build subnetwork set
subnets = []
for s in seed_nodes:
    subnets.append(get_ego_net_nodes_quvine(G, s, k=2, max_nodes=60))

extra_centers = importance_sample_centers(G, m=15, exclude=set(seed_nodes), bins=6, seed=7)
for c in extra_centers:
    subnets.append(get_ego_net_nodes_quvine(G, c, k=2, max_nodes=60))

print('#subnetworks:', len(subnets), 'avg size:', float(np.mean([len(S) for S in subnets])))

## 3) Quantum-walk simulation using QuVINE's CTQW module

We use QuVINE's `generate_ctqw_hiperwalk_scores` function to run continuous-time quantum walks on each induced subgraph.

In [ ]:
def get_subgraph_and_mapping(G, nodes):
    """
    Get induced subgraph using QuVINE's induce_subgraph_by_nodes.
    
    Args:
        G: NetworkX graph
        nodes: List of nodes
    
    Returns:
        Tuple of (subgraph, node_to_index_mapping, nodes_list)
    """
    nodes = list(nodes)
    idx = {n: i for i, n in enumerate(nodes)}
    H = induce_subgraph_by_nodes(G, set(nodes))
    return H, idx, nodes


def classical_heat_prob(A_sub, start_local, t=1.0):
    """
    Fallback: classical heat diffusion on subgraph Laplacian.
    Used when Hiperwalk is not available.
    """
    from scipy.linalg import expm
    n = A_sub.shape[0]
    deg = A_sub.sum(axis=1)
    Ls = np.diag(deg) - A_sub
    e0 = np.zeros(n)
    e0[start_local] = 1.0
    v = expm(-t*Ls) @ e0
    v = np.maximum(v, 0)
    return v / v.sum() if v.sum() > 0 else v


# Compute local quantum targets p_Q^S using QuVINE's CTQW function
q_targets = []
for S in subnets:
    center = S[0]
    H_sub, idx, nodes_list = get_subgraph_and_mapping(G, S)
    
    if HIPERWALK_AVAILABLE:
        # Use QuVINE's generate_ctqw_hiperwalk_scores function
        try:
            scores = generate_ctqw_hiperwalk_scores(
                G=G,
                root=center,
                view_nodes=nodes_list,
                steps=25,
                gamma=1.0
            )
            # Convert scores dict to probability array in node order
            pQ = np.array([scores.get(n, 0.0) for n in nodes_list])
            pQ = pQ / pQ.sum() if pQ.sum() > 0 else pQ
        except Exception as e:
            print(f"Warning: CTQW failed for center {center}, using classical fallback. Error: {e}")
            A_sub = nx.to_numpy_array(H_sub, nodelist=nodes_list, dtype=float)
            start_local = idx[center]
            pQ = classical_heat_prob(A_sub, start_local, t=1.0)
    else:
        # Fallback to classical heat diffusion
        A_sub = nx.to_numpy_array(H_sub, nodelist=nodes_list, dtype=float)
        start_local = idx[center]
        pQ = classical_heat_prob(A_sub, start_local, t=1.0)

    q_targets.append({'nodes': nodes_list, 'center': center, 'pQ': pQ})

print('Computed targets for', len(q_targets), 'subnetworks')
print('Example target length:', len(q_targets[0]['pQ']))

## 4) Fit global filter parameters $\theta$

We define a *global* operator $g_{\theta}(L)$ on the **full graph** and compare its output restricted to subnetwork nodes.

### 4A) Heat-kernel parameter $t$
$g_t(L)=\exp(-tL)$, fit by grid search.

In [ ]:
def expm_multiply_restricted(L, t, x, node_indices, node_to_idx):
    """
    Compute y = exp(-tL) x and return y restricted to specified nodes.
    
    Args:
        L: Laplacian matrix (sparse)
        t: Time parameter
        x: Initial vector
        node_indices: List of node IDs to restrict output
        node_to_idx: Dictionary mapping node IDs to matrix indices
    
    Returns:
        Restricted output vector
    """
    y = spla.expm_multiply((-t) * L, x)
    # Convert node IDs to matrix indices
    idx_list = [node_to_idx[n] for n in node_indices]
    return np.asarray(y)[idx_list]


def fit_heat_time(L, q_targets, t_grid, node_to_idx, loss='l2'):
    """
    Fit heat kernel time parameter by matching quantum walk targets.
    
    Args:
        L: Laplacian matrix
        q_targets: List of quantum walk target distributions
        t_grid: Grid of time values to search
        node_to_idx: Dictionary mapping node IDs to matrix indices
        loss: Loss function ('l2' or 'kl')
    
    Returns:
        Tuple of (best_loss, best_t)
    """
    best_loss, best_t = np.inf, None
    N = L.shape[0]

    for t in t_grid:
        tot = 0.0
        for item in q_targets:
            nodes = item['nodes']
            center = item['center']
            pQ = item['pQ']

            x = np.zeros(N)
            x[node_to_idx[center]] = 1.0

            yS = expm_multiply_restricted(L, t, x, nodes, node_to_idx)
            yS = np.maximum(yS, 0)
            pT = yS / yS.sum() if yS.sum() > 0 else yS

            if loss == 'l2':
                tot += np.sum((pT - pQ) ** 2)
            elif loss == 'kl':
                eps = 1e-12
                tot += np.sum(pQ * (np.log(pQ + eps) - np.log(pT + eps)))
            else:
                raise ValueError('Unknown loss')

        if tot < best_loss:
            best_loss, best_t = tot, t

    return best_loss, best_t


t_grid = np.linspace(0.1, 5.0, 20)
loss_val, t_star = fit_heat_time(L, q_targets, t_grid, node_to_idx, loss='l2')
print('Best t:', t_star, 'loss:', loss_val)

### 4B) Polynomial filter coefficients $\{a_k\}$

Approximate $g(L)$ by $g_{\theta}(L)=\sum_{k=0}^K a_k L^k$ and fit $a_k$ by least squares on the restricted subnetwork targets.

In [ ]:
def apply_poly_filter(L, coeffs, x):
    """
    Compute y = sum_k a_k L^k x using sparse matvecs.
    
    Args:
        L: Laplacian matrix
        coeffs: Polynomial coefficients
        x: Input vector
    
    Returns:
        Filtered output vector
    """
    y = coeffs[0] * x
    v = x.copy()
    for k in range(1, len(coeffs)):
        v = L @ v
        y = y + coeffs[k] * v
    return np.asarray(y)


def fit_poly_coeffs(L, q_targets, node_to_idx, K=4, ridge=1e-6):
    """
    Fit polynomial filter coefficients by least squares.
    
    Args:
        L: Laplacian matrix
        q_targets: List of quantum walk target distributions
        node_to_idx: Dictionary mapping node IDs to matrix indices
        K: Polynomial degree
        ridge: Ridge regularization parameter
    
    Returns:
        Array of polynomial coefficients
    """
    AtA = np.zeros((K + 1, K + 1))
    Atb = np.zeros(K + 1)
    N = L.shape[0]

    for item in q_targets:
        nodes = item['nodes']
        center = item['center']
        pQ = item['pQ']

        x = np.zeros(N)
        x[node_to_idx[center]] = 1.0

        basis = []
        v = x.copy()
        # Convert node IDs to indices for extraction
        node_indices = [node_to_idx[n] for n in nodes]
        basis.append(v[node_indices])
        for _ in range(1, K + 1):
            v = L @ v
            basis.append(np.asarray(v)[node_indices])

        Phi = np.stack(basis, axis=1)  # |S| x (K+1)
        b = pQ

        AtA += Phi.T @ Phi
        Atb += Phi.T @ b

    AtA += ridge * np.eye(K + 1)
    return np.linalg.solve(AtA, Atb)


poly_coeffs = fit_poly_coeffs(L, q_targets, node_to_idx, K=4, ridge=1e-6)
print('Polynomial coeffs:', poly_coeffs)

## 5) Apply calibrated operator to node features and train a simple model

We compute $Z = g_{\theta}(L)X$ and train a lightweight classifier.

Replace the toy features/labels with real omics features and disease-gene labels.

In [ ]:
# Toy node features (replace with real features)
F = 16
X = np.random.normal(size=(N, F))

# Toy labels: mark seeds as positives (replace with known disease genes)
y = np.zeros(N, dtype=int)
for seed in seed_nodes:
    y[node_to_idx[seed]] = 1

# Apply heat filter with t_star
Z_heat = np.zeros_like(X)
for f in range(F):
    Z_heat[:, f] = spla.expm_multiply((-t_star) * L, X[:, f])

# Apply polynomial filter
Z_poly = np.zeros_like(X)
for f in range(F):
    Z_poly[:, f] = apply_poly_filter(L, poly_coeffs, X[:, f])

print('Z_heat shape:', Z_heat.shape, 'Z_poly shape:', Z_poly.shape)

### Train/evaluate (quick sanity check)

This uses logistic regression if `scikit-learn` is available.

In [ ]:
if not SKLEARN_AVAILABLE:
    print('scikit-learn not available; skipping classifier demo')
else:
    rng = np.random.default_rng(0)
    idx = np.arange(N)
    rng.shuffle(idx)
    split = int(0.8 * N)
    tr, te = idx[:split], idx[split:]

    def eval_model(Z, name):
        clf = LogisticRegression(max_iter=2000)
        clf.fit(Z[tr], y[tr])
        p = clf.predict_proba(Z[te])[:, 1]
        auc = roc_auc_score(y[te], p)
        ap = average_precision_score(y[te], p)
        print(f'{name}: AUROC={auc:.3f}, AUPRC={ap:.3f}')

    eval_model(X, 'Raw features')
    eval_model(Z_heat, f'Heat-filtered (t={t_star:.2f})')
    eval_model(Z_poly, 'Poly-filtered')

## 6) Applying Q-Caliber to Real Biological Networks

To use Q-Caliber on real PPI networks for disease gene prioritization:

1. **Load PPI graph**: Use `quvine.data.data_loader.load_graph()` with a config file
2. **Load disease seeds/targets**: Use `quvine.data.data_loader.load_seeds_and_targets()`
3. **Run Q-Caliber pipeline**:
   - Sample subnetworks around seeds and diverse nodes
   - Generate quantum walk targets with `generate_ctqw_hiperwalk_scores()`
   - Calibrate global diffusion parameters (heat kernel or polynomial)
   - Apply calibrated operator to full graph
4. **Benchmark**: Compare Q-Caliber against:
   - Classical diffusion: RWR, heat kernel, PageRank
   - GNN methods: APPNP/PPNP, GDC
   - Pure quantum walks (if computationally feasible)

## Q-Caliber Advantages

- **Scalability**: Quantum walks only on small subgraphs, classical diffusion on full graph
- **Quantum-informed**: Captures quantum walk behavior in global operator
- **Flexibility**: Works with any graph diffusion operator (heat kernel, polynomial, etc.)
- **Integration**: Seamlessly integrates with QuVINE's existing pipeline

## Key QuVINE Functions Used

- `quvine.data.subgraph.expand_neighborhood()`: K-hop neighborhood expansion
- `quvine.data.subgraph.induce_subgraph_by_nodes()`: Induced subgraph creation
- `quvine.walks.ctqw.generate_ctqw_hiperwalk_scores()`: CTQW simulation with Hiperwalk
- `quvine.utils.utilities.get_stats()`: Graph statistics computation

## References

- **Hiperwalk**: Quantum walk simulator - https://hiperwalk.org/
- **QuVINE**: This project's library in `src/quvine/`
- **Q-Caliber**: Quantum-Calibrated Graph Diffusion (this notebook)